<a href="https://colab.research.google.com/github/filipchudzynski/stock-market-non-gaussianity-analyzer_v2/blob/main/wavelets_log_energy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! git clone https://github.com/filipchudzynski/stock-market-non-gaussianity-analyzer_v2.git

Cloning into 'stock-market-non-gaussianity-analyzer_v2'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 178 (delta 66), reused 96 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 25.62 MiB | 6.38 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [33]:
from IPython.display import Javascript
display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 5000})'''))



<IPython.core.display.Javascript object>

In [2]:
import sys
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/wavelet_log_energy_pipeline/")

In [13]:
#example_pipeline.py
from toy_models import white_noise, brownian_motion
from wavelet_transform import compute_cwt
from energy_field import wavelet_energy, log_energy_field
from intermittency import intermittency_variance
from cascade_memory import log_energy_covariance, estimate_lambda2
from scale_coupling import scale_mutual_information



Intermittency spectrum: [4.91135036 4.73132805 4.89561734 4.90872237 5.11483978 4.86985448
 4.864279   4.80593889 4.98492283 4.94515951]
Estimated lambda^2: [np.float64(0.08582226658229758), np.float64(0.11169975702904023), np.float64(0.14303287821074354), np.float64(0.18945935230666375), np.float64(0.23888068436639392), np.float64(0.27260361248270126), np.float64(0.2972118861960182), np.float64(0.29643856736761764), np.float64(0.3267844591360978), np.float64(0.3787206903990725), np.float64(0.4104703796426114), np.float64(0.3748120554142585), np.float64(0.38061276973942404), np.float64(0.4070097098378636), np.float64(0.4084058949522225), np.float64(0.4285799028122501), np.float64(0.4482420413407071), np.float64(0.4797908326445705), np.float64(0.49332042086768774), np.float64(0.5807736962834961), np.float64(0.6062230366644603), np.float64(0.6230440328208026), np.float64(0.6069902330965771), np.float64(0.6087361368469042), np.float64(0.6027789950185205), np.float64(0.5846402653987507), n

In [34]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def validate_lambda2_fits(L, n_examples=5):
    """
    L : log-energy field, shape (n_scales, n_times)
    n_examples : number of scales to visualize
    """

    n_scales = L.shape[0]
    example_scales = np.linspace(0, n_scales - 1, n_examples, dtype=int)

    fig = make_subplots(
        rows=n_examples,
        cols=1,
        subplot_titles=[f"Scale index {s}" for s in example_scales]
    )

    row = 1
    for s in example_scales:

        # compute covariance
        U = L[s]
        cov = log_energy_covariance(U)
        lags = np.arange(1, len(cov) + 1)

        # compute polyfit
        x = np.log(lags)
        slope, intercept = np.polyfit(x, cov, 1)
        y_fit = slope * x + intercept
        lam2 = -slope

        # plot data + fit
        fig.add_trace(
            go.Scatter(
                x=x,
                y=cov,
                mode="markers",
                name=f"scale {s}",
                marker=dict(size=5)
            ),
            row=row, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=x,
                y=y_fit,
                mode="lines",
                name=f"fit λ²={lam2:.3f}",
                line=dict(width=2)
            ),
            row=row, col=1
        )

        row += 1

    fig.update_layout(
        height=300 * n_examples,
        width=900,
        title="Validation of λ² estimation: covariance vs log(lag) with polyfit"
    )

    fig.show()


In [41]:
def validate_pipeline(signal=white_noise):

    # -----------------------------
    # Generate signal
    # -----------------------------
    x = signal(20000)

    # -----------------------------
    # Scales
    # -----------------------------
    scales = np.arange(2, 128)

    # -----------------------------
    # Wavelet transform
    # -----------------------------
    W = compute_cwt(x, scales)   # user-provided

    # -----------------------------
    # Energy
    # -----------------------------
    E = wavelet_energy(W)

    # -----------------------------
    # Log-energy field
    # -----------------------------
    L = log_energy_field(E)

    # -----------------------------
    # Intermittency spectrum
    # -----------------------------
    I = intermittency_variance(L)
    print("Intermittency spectrum (first 10):", I[:10])

    # -----------------------------
    # Cascade memory (lambda^2)
    # -----------------------------
    lam2 = []
    for l in L:
        cov = log_energy_covariance(l)
        lags = np.arange(1, len(cov)+1)
        lam2.append(estimate_lambda2(cov, lags))


    print("Estimated lambda^2 (first 10):", lam2[:10])

    # ============================================================
    # PLOTS
    # ============================================================

    # 1. Raw signal
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(y=x, mode="lines"))
    fig1.update_layout(title="Raw Signal (White Noise)")
    fig1.show()

    # 2. Wavelet energy heatmap
    # fig2 = go.Figure(data=go.Heatmap(
    #     z=E,
    #     x=np.arange(E.shape[1]),
    #     y=scales,
    #     colorscale="Viridis"
    # ))
    # fig2.update_layout(title="Wavelet Energy")
    # fig2.show()

    # # 3. Log-energy field heatmap
    # fig3 = go.Figure(data=go.Heatmap(
    #     z=L,
    #     x=np.arange(L.shape[1]),
    #     y=scales,
    #     colorscale="Turbo"
    # ))
    # fig3.update_layout(title="Log-Energy Field")
    # fig3.show()

    # 4. Intermittency spectrum
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(x=scales, y=I, mode="lines+markers"))
    fig4.update_layout(title="Intermittency Spectrum (Variance of Log-Energy)")
    fig4.show()

    # 5. Lambda^2 across scales
    fig5 = go.Figure()
    fig5.add_trace(go.Scatter(x=scales, y=lam2, mode="lines+markers"))
    fig5.update_layout(title="Estimated λ² Across Scales")
    fig5.show()

    # 6. Example covariance plot
    example_scale = 20
    cov = log_energy_covariance(L[example_scale])
    fig6 = go.Figure()
    fig6.add_trace(go.Scatter(x=np.arange(1,len(cov)+1), y=cov, mode="lines"))
    fig6.update_layout(title=f"Log-Energy Covariance at Scale {scales[example_scale]}")
    fig6.show()

    validate_lambda2_fits(L,5)



# ============================================================
# RUN VALIDATION
# ============================================================



In [42]:
validate_pipeline(signal=white_noise)

Intermittency spectrum (first 10): [5.07791439 4.9587044  4.83431031 4.97913779 5.01348716 4.88556247
 4.95928363 4.94319741 5.0056389  4.98860088]
Estimated lambda^2 (first 10): [np.float64(0.06390814810539432), np.float64(0.11293891722056643), np.float64(0.12241736456868015), np.float64(0.1763254523870806), np.float64(0.19417174828665748), np.float64(0.19791587649556763), np.float64(0.26682288533802395), np.float64(0.30491482300410677), np.float64(0.32657452277295795), np.float64(0.3686744015155935)]
